In [125]:
import sys
dir_git = "/Users/usuario/git/sisepuede"
if dir_git not in sys.path:
    sys.path.append(dir_git)
    
import importlib
import matplotlib.pyplot as plt
import numpy as np
import os, os.path
import pandas as pd
import pathlib
import sisepuede.core.support_classes as sc
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
import sisepuede.utilities._toolbox as sf
import warnings
from typing import *
warnings.filterwarnings("ignore")

import utils.common_data_needs as cdn



# **TODO**: use max/min to reflect historical land use values
- need to review frac values

# Set bounds on area
- We require that flooded areas only move within +/- 5% of historical

In [ ]:
import sys
dir_git = "/Users/usuario/git/sisepuede"
if dir_git not in sys.path:
    sys.path.append(dir_git)
    
import importlib
import matplotlib.pyplot as plt
import numpy as np
import os, os.path
import pandas as pd
import pathlib
import sisepuede.core.support_classes as sc
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
import sisepuede.utilities._toolbox as sf
import warnings
warnings.filterwarnings("ignore")

import utils.common_data_needs as cdn



In [3]:
dict_ssp = cdn._setup_sisepuede_elements()

matt = dict_ssp.get("model_attributes", )
models = dict_ssp.get("models", )
regions = dict_ssp.get("regions", )
time_periods = dict_ssp.get("time_periods", )

In [166]:
##  SET SOME GLOBALS FOR ANALYSIS

# model variables
_MODVAR_AREA_MAX = matt.get_variable("Maximum Area")
_MODVAR_AREA_MIN = matt.get_variable("Minimum Area")
_MODVAR_AREA = matt.get_variable("Area of Region")
_MODVAR_ILUP = matt.get_variable("Initial Land Use Area Proportion")

# set some derivative file names
_FILE_NAME_AREA_MAX = cdn.file_name_from_variable(_MODVAR_AREA_MAX, )
_FILE_NAME_AREA_MIN = cdn.file_name_from_variable(_MODVAR_AREA_MIN, )


# units managers
_UM_AREA = matt.get_unit("area")


# get base data, ignoring the inputs constructed herein
df_uganda = cdn._build_from_outputs(
    (
        min(time_periods.all_years),
        max(time_periods.all_years)
    ),
    fns_exclude = [
        _FILE_NAME_AREA_MAX,
        _FILE_NAME_AREA_MIN,
    ], 
    force_complete_build = True,
    merge_type = "outer",
    print_info = False,
    stop_on_error = True, 
)


# get base data frames from base dataset; we'll overwrite data below, then export it
_DF_AREA_MAX = cdn.get_variable_df_for_overwrite(
    df_uganda, 
    _MODVAR_AREA_MAX,
)

_DF_AREA_MIN = cdn.get_variable_df_for_overwrite(
    df_uganda, 
    _MODVAR_AREA_MIN,
)

# Get UBOS data
- Data are sourced from [UBOS Land Statistics](https://www.ubos.org/explore-statistics/14/)


In [167]:
path_land_prev = cdn._PATH_INPUTS.joinpath("ubos", "National_Land_Cover_statistics_(sq._km)_Land.xlsx")
path_forest_reserve = cdn._PATH_INPUTS.joinpath("ubos", "Share_of_total_area_under_forest_reserves_by_region,_2015.xlsx")

##  set some data variables

# categories
_CAT_LNDU_FLOODED = "flooded"
_CAT_LNDU_FRST_PRIM = "forests_primary"
_CAT_LNDU_FRST_SCND = "forests_secondary"

# units from data
_UNITS_AREA_FOREST_RESERVE = "ha"
_UNITS_AREA_LAND_PREV = "km2"


# years to build
df_years = pd.DataFrame(
    {
        time_periods.field_year: range(
            cdn._YEARS_DEFAULT_MIN,
            cdn._YEARS_DEFAULT_MAX + 1,
        )
    }
)




def _check_area_bounds(
    df_base: pd.DataFrame,
    area_bounds_max: Union[float, None],
    area_bounds_min: Union[float, None],
    cat_lndu_check: str,
    return_area_init: str = "none",
) -> None:
    """Verify that 

    Function Arguments
    ------------------
    df_base : pd.DataFrame
        DataFrame containing SISEPUEDE inputs
    area_bounds_max : Union[float, None]
        Maximum area bound to be specified in terms of _MODVAR_AREA_MAX
        area units. If None, no bound is checked.
    area_bounds_min : Union[float, None]
        Minimum area bound to be specified in terms of _MODVAR_AREA_MIN
        area units. If None, no bound is checked.
    cat_lndu_check : str
        Land Use category to verify

    Keyword Arguments
    -----------------
    return_area_init : str
        Set whether or not to return the initial area
            * "none":     do not return the initial area
            * "area":     return the initial area in terms of _MODVAR_AREA
            * "area_max": return the initial area in terms of _MODVAR_AREA_MAX
            * "area_min": return the initial area in terms of _MODVAR_AREA_MIN
            
    """

    ##  INITIALIZE 
    
    # get some units (printed later)
    units_modvar_area = _MODVAR_AREA.attribute("unit_area")
    units_modvar_area_max = _MODVAR_AREA_MAX.attribute("unit_area")
    units_modvar_area_min = _MODVAR_AREA_MIN.attribute("unit_area")
    
    # start by getting initial areas from base data frame
    area_0 = float(_MODVAR_AREA.get_from_dataframe(df_base).iloc[0])
    
    # convert to comparable terms
    area_0_terms_max = area_0*_UM_AREA.convert(
        units_modvar_area,
        units_modvar_area_max,
    )
    area_0_terms_min = area_0*_UM_AREA.convert(
        units_modvar_area,
        units_modvar_area_min,
    )

    # get field and associated max/min comparisons (the initial area in terms of each bound's units)
    field_prop = _MODVAR_ILUP.build_fields(
        category_restrictions = cat_lndu_check,
    )
    
    area_0_base = area_0*float(df_uganda[field_prop].iloc[0])
    area_cat_init_terms_max = (
        area_0_terms_max*float(df_uganda[field_prop].iloc[0])
    )
    area_cat_init_terms_min = (
        area_0_terms_min*float(df_uganda[field_prop].iloc[0])
    )

    # return area?
    if return_area_init in ["area", "area_max", "area_min"]:
        out = (
            {
                "area": area_0_base,
                "area_max": area_cat_init_terms_max,
                "area_min": area_cat_init_terms_min,
            }
            .get(return_area_init)
        )

        return out

    

    ##  NOTIFY AND CHECK
    
    # check that initial conditions are not violated--start with max
    if sf.isnumber(area_bounds_max):
        print(f"Checking maximum area ({units_modvar_area_max}):")
        print(f"\tInitial area: {area_cat_init_terms_max}")
        print(f"\tCandidate bound: {area_bounds_max}")
        
        if area_cat_init_terms_max > area_bounds_max:
            raise RuntimeError(f"Conflict in maxima")

    # verify minimum is OK
    if sf.isnumber(area_bounds_min):
        print(f"Checking minimum area ({units_modvar_area_min}):")
        print(f"\tInitial area: {area_cat_init_terms_min}")
        print(f"\tCandidate bound: {area_bounds_min}")
        
        if area_cat_init_terms_min < area_bounds_min:
            raise RuntimeError(f"Conflict in minima")

    
    return None






##  Work on open water bounds data

In [168]:
# read data frame
df_land_prev = (
    pd.read_excel(
        path_land_prev, 
        nrows = 8,
        skiprows = 3,
    )
    .drop(
        columns = ["Unnamed: 0"]
    )
)


# get scalars
scalar_land_prev_to_max_area = _UM_AREA.convert(
    _UNITS_AREA_LAND_PREV,
    _MODVAR_AREA_MAX.attribute("unit_area")
)

scalar_land_prev_to_min_area = _UM_AREA.convert(
    _UNITS_AREA_LAND_PREV,
    _MODVAR_AREA_MIN.attribute("unit_area")
)

# get open water variance
field_lu = "Land Use/Land Cover Type"
vec_areas = (
    df_land_prev[
        df_land_prev[field_lu].isin(["Open water"])
    ]
    .drop(columns = field_lu, )
    .to_numpy()[0]
)

area_mean = np.mean(vec_areas)
dev_max, dev_min = vec_areas.max()/area_mean - 1, 1 - vec_areas.min()/area_mean

# round to nearest 0.05
factor = 1/0.05
dev_max_rd = np.ceil(dev_max*factor)/factor
dev_min_rd = np.ceil(dev_min*factor)/factor

# get a max and min area
area_max = area_mean*(1 + dev_max_rd)*scalar_land_prev_to_max_area
area_min = area_mean*(1 - dev_min_rd)*scalar_land_prev_to_min_area
_check_area_bounds(
    df_uganda, 
    area_max,
    area_min,
    _CAT_LNDU_FLOODED,
)


##  BUILD THE DFS

# maximum area
field_max = _MODVAR_AREA_MAX.build_fields(
    category_restrictions = _CAT_LNDU_FLOODED,
)
df_max_flooded = df_years.copy()
df_max_flooded[field_max] = area_max

# minimum area
field_min = _MODVAR_AREA_MIN.build_fields(
    category_restrictions = _CAT_LNDU_FLOODED,
)
df_min_flooded = df_years.copy()
df_min_flooded[field_min] = area_min



Checking maximum area (ha):
	Initial area: 3737645.3190834997
	Candidate bound: 3898020.0000000005
Checking minimum area (ha):
	Initial area: 3737645.3190834997
	Candidate bound: 3526779.9999999995


###  ovewrite global dfs

In [169]:


_DF_AREA_MIN = sf.match_df_to_target_df(
    _DF_AREA_MIN,
    df_min_flooded,
    fields_index = [time_periods.field_year],
    overwrite_only = True,
)

_DF_AREA_MAX = sf.match_df_to_target_df(
    _DF_AREA_MAX,
    df_max_flooded,
    fields_index = [time_periods.field_year],
    overwrite_only = True,
)


##  Add in forest reserves, which only bound lower end
- Apply to primary forests by assumption

In [180]:
df_forest_reserves = pd.read_excel(
    path_forest_reserve,
    skiprows = 2, 
)

# 
area_forest_reserves = float(
    df_forest_reserves[
        df_forest_reserves["Region"].isin(["Uganda"])
    ]
    .get("Total Forest reserve Area")
    .iloc[0]
)

# scale to appropriate units
area_forest_reserves *= _UM_AREA.convert(
    _UNITS_AREA_FOREST_RESERVE,
    _MODVAR_AREA_MIN.attribute("unit_area")
)


# check the min specification
area_max = area_mean*(1 + dev_max_rd)*scalar_land_prev_to_max_area
area_min = area_mean*(1 - dev_min_rd)*scalar_land_prev_to_min_area
area_init_primary = _check_area_bounds(
    df_uganda, 
    None,
    area_forest_reserves,
    _CAT_LNDU_FRST_PRIM,
    return_area_init = "area_min",
)

# for now, assume 80% of primary forests are covered... need geospatial data to do this
area_forest_reserves_primary = (
    0.8*area_init_primary
    if area_init_primary < area_forest_reserves
    else area_forest_reserves
)


area_forest_reserves_secondary = area_forest_reserves - area_forest_reserves_primary



# CHECK BOTH BOUNDS
_check_area_bounds(
    df_uganda, 
    None,
    area_forest_reserves_primary,
    _CAT_LNDU_FRST_PRIM,
)

_check_area_bounds(
    df_uganda, 
    None,
    area_forest_reserves_secondary,
    _CAT_LNDU_FRST_SCND,
)



##  BUILD DFS

# minimum area
field_min_prim = _MODVAR_AREA_MIN.build_fields(
    category_restrictions = _CAT_LNDU_FRST_PRIM,
)

df_min_forest = df_years.copy()
df_min_forest[field_min_prim] = area_forest_reserves_primary


if area_forest_reserves_secondary > 0:
    field_min_scnd = _MODVAR_AREA_MIN.build_fields(
        category_restrictions = _CAT_LNDU_FRST_SCND,
    )
    df_min_forest[field_min_scnd] = area_forest_reserves_secondary


Checking minimum area (ha):
	Initial area: 183076.26516699823
	Candidate bound: 146461.0121335986
Checking minimum area (ha):
	Initial area: 1794254.9865739993
	Candidate bound: 383133.98786640144


##  Overwrite in global DF

In [182]:
_DF_AREA_MIN = sf.match_df_to_target_df(
    _DF_AREA_MIN,
    df_min_forest,
    fields_index = [time_periods.field_year],
    overwrite_only = True,
)

# Export tables

In [193]:
_DF_AREA_MAX.to_csv(
    cdn._PATH_OUTPUTS.joinpath(_FILE_NAME_AREA_MAX),
    encoding = "UTF-8",
    index = None,
)

_DF_AREA_MIN.to_csv(
    cdn._PATH_OUTPUTS.joinpath(_FILE_NAME_AREA_MIN),
    encoding = "UTF-8",
    index = None,
)